In [ ]:
# Import required libraries
import cv2
import numpy as np
from ultralytics import YOLO
import time

print("Libraries imported successfully!")
print("OpenCV version:", cv2.__version__)

print("Loading YOLOv8 model...")
model = YOLO('yolov8n.pt')
print("Model loaded successfully!")

In [ ]:
# Function to perform real-time object detection and tracking
def real_time_detection_tracking():
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Could not open camera")
        return
    
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    print("Camera initialized successfully!")
    print("Press 'q' to quit the application")
    
    frame_count = 0
    start_time = time.time()
    
    try:
        while True:
            ret, frame = cap.read()
            
            if not ret:
                print("Error: Failed to capture frame")
                break
            
            frame_count += 1
            
            # Perform tracking (YOLOv8 with built-in tracking)
            # track() method uses SORT algorithm internally for object tracking
            results = model.track(frame, persist=True, verbose=False)
            
            # Process results
            if results[0].boxes is not None:
                # Get boxes, classes, track IDs, and confidence scores
                boxes = results[0].boxes.xyxy.cpu().numpy()  # Bounding boxes
                classes = results[0].boxes.cls.cpu().numpy()  # Class indices
                confidences = results[0].boxes.conf.cpu().numpy()  # Confidence scores
                
                # Get track IDs if available
                track_ids = results[0].boxes.id
                if track_ids is not None:
                    track_ids = track_ids.cpu().numpy()
                
                # Draw bounding boxes and labels
                for i, (box, cls, conf) in enumerate(zip(boxes, classes, confidences)):
                    if conf > 0.5:  # Confidence threshold
                        x1, y1, x2, y2 = map(int, box)
                        
                        # Get class name
                        class_name = model.names[int(cls)]
                        
                        # Get track ID if available
                        track_id = int(track_ids[i]) if track_ids is not None else None
                        
                        # Create label with class name, confidence, and track ID
                        if track_id is not None:
                            label = f"{class_name} ID:{track_id} {conf:.2f}"
                            color = (0, 255, 0)  # Green for tracked objects
                        else:
                            label = f"{class_name} {conf:.2f}"
                            color = (0, 0, 255)  # Red for non-tracked objects
                        
                        # Draw bounding box
                        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                        
                        # Draw label background
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)[0]
                        cv2.rectangle(frame, (x1, y1 - label_size[1] - 10), 
                                    (x1 + label_size[0], y1), color, -1)
                        
                        # Draw label text
                        cv2.putText(frame, label, (x1, y1 - 5), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            # Calculate and display FPS
            elapsed_time = time.time() - start_time
            fps = frame_count / elapsed_time if elapsed_time > 0 else 0
            
            # Add FPS counter to frame
            cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            
            # Add instructions
            cv2.putText(frame, "Press 'q' to quit", (10, frame.shape[0] - 10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            # Display the frame
            cv2.imshow('YOLOv8 Real-Time Object Detection & Tracking', frame)
            
            # Check for 'q' key press to quit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
                
    except KeyboardInterrupt:
        print("\nStopping detection...")
    
    finally:
        # Release resources
        cap.release()
        cv2.destroyAllWindows()
        print("Camera released and windows closed.")
        print(f"Total frames processed: {frame_count}")
        print(f"Average FPS: {fps:.2f}")

print("Real-time detection function defined successfully!")

In [ ]:
print("Starting real-time object detection and tracking...")
print("Make sure your camera is connected and not being used by other applications.")
print("The application will open in a new window.")
print("\nInstructions:")
print("- Point your camera at different objects to see detection in action")
print("- Objects will be assigned unique tracking IDs")
print("- Press 'q' in the video window to stop the application")
print("\nStarting in 3 seconds...")

import time
time.sleep(3)

real_time_detection_tracking()